# COMP0005 - GROUP COURSEWORK
# Experimental Evaluation of Search Data Structures and Algorithms

The cell below defines **AbstractSearchInterface**, an interface to support basic insert/search operations; you will need to implement this three times, to realise your three search data structures of choice among: (1) *2-3 Tree*, (2) *AVL Tree*, (3) *LLRB BST*; (4) *B-Tree*; and (5) *Scapegoat Tree*. <br><br>**Do NOT modify the next cell** - use the dedicated cells further below for your implementation instead. <br>

In [22]:
# DO NOT MODIFY THIS CELL

from abc import ABC, abstractmethod  

class AbstractSearchInterface(ABC):
    '''
    Abstract class to support search/insert operations (plus underlying data structure)
    
    '''
        
    @abstractmethod
    def insertElement(self, element):     
        '''
        Insert an element in a search tree
            Parameters:
                    element: string to be inserted in the search tree (string)

            Returns:
                    "True" after successful insertion, "False" if element is already present (bool)
        '''
        
        pass 
    

    @abstractmethod
    def searchElement(self, element):
        '''
        Search for an element in a search tree
            Parameters:
                    element: string to be searched in the search tree (string)

            Returns:
                    "True" if element is found, "False" otherwise (bool)
        '''

        pass

Use the cell below to define any auxiliary data structure and python function you may need. Leave the implementation of the main API to the next code cells instead.

In [23]:
class ScapeGoatNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
        self.size = 1

        
class AVLNode:
    def __init__(self, value):
        self.value = value
        self.height = 1
        self.leftChild = None
        self.rightChild = None  
 

def AVLGetHeight(node):
    if node == None:
        return 0
    return node.height

def AVLGetBF(node):
    if node == None:
        return 0
    return AVLGetHeight(node.leftChild) - AVLGetHeight(node.rightChild)

def RightRotation(z):
    y = z.leftChild
    T3 = y.rightChild
    y.rightChild = z
    z.leftChild = T3
    z.height = 1 + max(AVLGetHeight(z.leftChild), AVLGetHeight(z.rightChild))
    y.height = 1 + max(AVLGetHeight(y.leftChild), AVLGetHeight(y.rightChild))
    return y

def LeftRotation(z):
    y = z.rightChild
    T2 = y.leftChild
    y.leftChild = z
    z.rightChild = T2
    z.height = 1 + max(AVLGetHeight(z.leftChild), AVLGetHeight(z.rightChild))
    y.height = 1 + max(AVLGetHeight(y.leftChild), AVLGetHeight(y.rightChild))
    return y

def AVLBalance(root, element):
    bf = AVLGetBF(root)
    if bf > 1:
        if element < root.leftChild.value:
            root = RightRotation(root)
        else:
            root.leftChild = LeftRotation(root.leftChild)
            root = RightRotation(root)
    elif bf < -1:
        if element > root.rightChild.value:
            root = LeftRotation(root)
        else:
            root.rightChild = RightRotation(root.rightChild)
            root = LeftRotation(root)

    return root

def AVLInsert(root, element):
    if root == None:
        return AVLNode(element)
    
    if element < root.value:
        root.leftChild = AVLInsert(root.leftChild, element)
    elif element > root.value:
        root.rightChild = AVLInsert(root.rightChild, element)
    else:
        return root
    
    root.height = 1 + max(AVLGetHeight(root.leftChild), AVLGetHeight(root.rightChild))

    root = AVLBalance(root, element)
    return root

def AVLSearch(root, element):
    if root == None:
        return False
    if element == root.value:
        return True
    if element < root.value:
        return AVLSearch(root.leftChild, element)
    else:
        return AVLSearch(root.rightChild, element)


class TwoThreeNode:
    def __init__(self, key):
        self.keys = [key]
        self.children = []

    def is_leaf(self):
        return len(self.children) == 0

    def is_two_node(self):
        return len(self.keys) == 1

    def is_four_node(self):
        return len(self.keys) == 3


Use the cell below to implement the requested API by means of **2-3 Tree** (if among your chosen data structure).

In [24]:
class TwoThreeTree(AbstractSearchInterface):
    def __init__(self):
        self._root: TwoThreeNode | None = None
        self._size: int = 0

    def insertElement(self, element: str) -> bool:

        inserted = False

        if self.searchElement(element):
            return False                            # duplicate: no-op

        if self._root is None:
            self._root = TwoThreeNode(element)
            self._size += 1
            inserted = True
            return inserted

        # _insert returns (mid_key, left, right) only when the root splits.
        overflow = self._insert(self._root, element)
        if overflow is not None:
            mid_key, left, right = overflow
            new_root = TwoThreeNode(mid_key)
            new_root.children = [left, right]
            self._root = new_root

        self._size += 1
        inserted = True
        return inserted

    def searchElement(self, element: str) -> bool:
        found = self._search(self._root, element)
        return found

    def _search(self, node: TwoThreeNode | None, element: str) -> bool:

        if node is None:
            return False
        if element in node.keys:
            return True
        if node.is_leaf():
            return False

        if element < node.keys[0]:
            return self._search(node.children[0], element)
        if node.is_two_node() or element < node.keys[1]:
            return self._search(node.children[1], element)
        return self._search(node.children[2], element)

    def _insert(self, node: TwoThreeNode, element: str):
        if node.is_leaf():
            self._add_key(node, element)
            if node.is_four_node():
                return self._split(node)
            return None

        child_idx = self._child_index(node, element)
        overflow = self._insert(node.children[child_idx], element)

        if overflow is None:
            return None

        mid_key, left, right = overflow
        self._add_key(node, mid_key)

        node.children.pop(child_idx)
        node.children.insert(child_idx, right)
        node.children.insert(child_idx, left)

        if node.is_four_node():
            return self._split(node)
        return None

    def _child_index(self, node: TwoThreeNode, element: str) -> int:
        if element < node.keys[0]:
            return 0
        if node.is_two_node() or element < node.keys[1]:
            return 1
        return 2

    @staticmethod
    def _add_key(node: TwoThreeNode, key: str) -> None:
        node.keys.append(key)
        node.keys.sort()

    @staticmethod
    def _split(node: TwoThreeNode):
        k0, k1, k2 = node.keys

        left = TwoThreeNode(k0)
        right = TwoThreeNode(k2)

        if not node.is_leaf():
            left.children  = node.children[:2]
            right.children = node.children[2:]

        return k1, left, right


Use the cell below to implement the requested API by means of **AVL Tree** (if among your chosen data structure).

In [25]:
class AVLTree(AbstractSearchInterface):
    def __init__(self):
        self.root = None
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
        if not self.searchElement(element):
            self.root = AVLInsert(self.root, element)
            inserted = True
        
        return inserted
    
    
    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE
        found = AVLSearch(self.root, element)
        return found  

Use the cell below to implement the requested API by means of **LLRB BST** (if among your chosen data structure).

In [26]:
class LLRBBST(AbstractSearchInterface):
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
      
        
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE

        
        return found  

Use the cell below to implement the requested API by means of **B-Tree** (if among your chosen data structure).

In [27]:
class BTree(AbstractSearchInterface):
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
      
        
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE

        
        return found

Use the cell below to implement the requested API by means of **Scapegoat Tree** (if among your chosen data structure).

In [28]:
class ScapegoatTree(AbstractSearchInterface):
    def __init__(self):
        self.root = None
        self.size = 0
        self.alpha = 2/3

    def insertElement(self, element):
        inserted = False
        current = self.root
        path = []
        direction = ""
        if self.root is None:
            self.root = ScapeGoatNode(element)
            self.size += 1
            return True
        while not inserted:
            if current is None:
                if direction == "left":
                    path[-1].left = ScapeGoatNode(element)
                elif direction == "right":
                    path[-1].right = ScapeGoatNode(element)
                self.size += 1
                inserted = True
            else:
                path.append(current)
                if current.key == element:
                     return False
                elif current.key > element:
                    direction = "left"
                    current = current.left
                elif current.key < element:
                    direction = "right"
                    current = current.right
        
        for node in reversed(path):
            node.size += 1
        
        scapegoat = self.findScapeGoat(path)
        if scapegoat is not None:
            nodes = self.getSubtreeNodes(scapegoat)
            balanced_subtree =  self.balanceSubtree(nodes)
            if scapegoat == self.root:
                self.root = balanced_subtree
            else:
                parent = path[path.index(scapegoat) - 1]
                if parent.left == scapegoat:
                    parent.left = balanced_subtree
                elif parent.right == scapegoat:
                    parent.right = balanced_subtree
        return inserted

    def searchElement(self, element):
        found = False
        if self.root is None:
            return False
        current = self.root
        while not found:
            if current is None:
                return False
            elif current.key == element:
                found = True
            elif current.key > element:
                current = current.left
            elif current.key < element:
                current = current.right
        return found

    def findSubtreeSize(self, node):
        if node is None:
            return 0

        return 1 + self.findSubtreeSize(node.left) + self.findSubtreeSize(node.right)

    def findScapeGoat(self, path):
        scapegoat = None
        for node in reversed(path):
            total = node.size
            if ((node.left and node.left.size > self.alpha * total)
                    or (node.right and node.right.size > self.alpha * total)):
                scapegoat = node
                break
        return scapegoat

    def getSubtreeNodes(self, node):
        if node is None:
            return []
        left_list = self.getSubtreeNodes(node.left)
        right_list = self.getSubtreeNodes(node.right)
        return left_list + [node] + right_list

    def balanceSubtree(self, nodes):
        if not nodes:
            return None
        middle = len(nodes) // 2
        left_list = nodes[:middle]
        right_list = nodes[(middle + 1):]
        root = nodes[middle]
        root.left = self.balanceSubtree(left_list)
        root.right = self.balanceSubtree(right_list)
        rightSize = root.right.size if root.right else 0
        leftSize = root.left.size if root.left else 0
        root.size = 1 + leftSize + rightSize

        return root

Use the cell below to implement the **synthetic data generator** needed by your experimental framework (be mindful of code readability and reusability).

In [29]:
import random
import string

class TestDataGenerator:
    
    def __init__(self, min_string_length=1, max_string_length=100, percentage_nearly_sorted=10, random_seed=None):
        self.min_string_length = min_string_length
        self.max_string_length = max_string_length
        self.percentage_nearly_sorted = percentage_nearly_sorted
        self.letters = string.ascii_lowercase
        
        if random_seed is not None:
            random.seed(random_seed)
    
    def generate_string_list(self, size):
        result = []
        for _ in range(size):
            length = random.randint(self.min_string_length, self.max_string_length)
            random_string = ''.join(random.choice(self.letters) for _ in range(length))
            result.append(random_string)
        return result
    
    def generate_insertion_order(self, data, pattern_type):
        result = data.copy()
        
        if pattern_type == 'random':
            random.shuffle(result)
        elif pattern_type == 'sorted':
            result = sorted(result)
        elif pattern_type == 'reverse_sorted':
            result = sorted(result, reverse=True)
        elif pattern_type == 'nearly_sorted':
            # Sort first, then make predetermined random swaps (default 10%)
            result = sorted(result)
            num_swaps = len(result) // self.percentage_nearly_sorted
            for _ in range(num_swaps):
                i = random.randint(0, len(result) - 1)
                j = random.randint(0, len(result) - 1)
                result[i], result[j] = result[j], result[i]
        else:
            raise ValueError(f"Unknown pattern_type: '{pattern_type}'. "
                           f"Must be one of: 'random', 'sorted', 'reverse_sorted', 'nearly_sorted'")
        
        return result

Use the cell below to implement the requested **experimental framework** (be mindful of code readability and reusability).

In [30]:
import timeit
import matplotlib.pyplot as plt

class ExperimentalFramework():
    
    @staticmethod
    def measure_time(func, *args, runs=1, **kwargs):
        result = None
        
        def wrapper():
            nonlocal result
            result = func(*args, **kwargs)
        
        timer = timeit.Timer(wrapper)
        elapsed_time = timer.timeit(number=runs) / runs
    
        return result, elapsed_time
    
    @staticmethod
    def run_single_experiment(structure_class, data, pattern_type, runs=1):
        size = len(data)
        structure_name = structure_class.__name__
        
        def do_insertions():
            tree = structure_class()
            for element in data:
                tree.insertElement(element)
            return tree
        
        tree, insert_time = ExperimentalFramework.measure_time(do_insertions, runs=runs)
        
        def do_searches():
            found_count = 0
            for element in data:
                if tree.searchElement(element):
                    found_count += 1
            return found_count
        
        found_count, search_time = ExperimentalFramework.measure_time(do_searches, runs=runs)
        
        return {
            'structure_name': structure_name,
            'size': size,
            'pattern': pattern_type,
            'insert_time': insert_time,
            'search_time': search_time,
            'found_count': found_count  # for verification, should be same as size value
        }
    
    @staticmethod
    def run_all_experiments(structure_classes, sizes, patterns, base_seed, runs=1):
        results = []

        total_experiments = len(structure_classes) * len(sizes) * len(patterns)
        experiment_count = 0

        print("=" * 70)
        print("RUNNING EXPERIMENTS")
        print("=" * 70)
        print(f"Runs per experiment: {runs}")
        print("=" * 70)

        for size in sizes:
            print(f"\n{'='*70}")
            print(f"SIZE = {size}")
            print(f"{'='*70}")

            for pattern in patterns:
                print(f"\nPattern: {pattern}")
                print("-" * 60)

                # Store averaged results per structure
                run_insert_times = {cls.__name__: [] for cls in structure_classes}
                run_search_times = {cls.__name__: [] for cls in structure_classes}

                # ---------------------------
                # RUN LOOP (NEW)
                # ---------------------------
                for run in range(runs):

                    seed = base_seed + run
                    
                    print(f"  Run {run+1}/{runs} (seed={seed})")

                    # NEW generator each run
                    run_generator = TestDataGenerator(random_seed=seed)

                    # Generate dataset ONCE per run
                    base_data = run_generator.generate_string_list(size)
                    ordered_data = run_generator.generate_insertion_order(
                        base_data, pattern
                    )

                    # Test ALL structures on SAME data
                    for structure_class in structure_classes:

                        structure_name = structure_class.__name__

                        result = ExperimentalFramework.run_single_experiment(
                            structure_class,
                            ordered_data,
                            pattern,
                            runs=1
                        )

                        run_insert_times[structure_name].append(result['insert_time'])
                        run_search_times[structure_name].append(result['search_time'])

                # ---------------------------
                # AVERAGE RESULTS
                # ---------------------------
                for structure_class in structure_classes:
                    experiment_count += 1
                    name = structure_class.__name__

                    avg_insert = sum(run_insert_times[name]) / runs
                    avg_search = sum(run_search_times[name]) / runs

                    print(f"[{experiment_count}/{total_experiments}] {name} "
                        f"✓ Insert: {avg_insert:.6f}s "
                        f"Search: {avg_search:.6f}s")

                    results.append({
                        'structure_name': name,
                        'size': size,
                        'pattern': pattern,
                        'insert_time': avg_insert,
                        'search_time': avg_search,
                        'found_count': size
                    })

        print("\n" + "=" * 70)
        print(f"EXPERIMENTS COMPLETE: {len(results)} results collected")
        print("=" * 70)

        return results
    
    @staticmethod
    def save_results(results, filename='experiment_results.json'):
        import json
        
        with open(filename, 'w') as f:
            json.dump(results, f, indent=2)
        
        print(f"✓ Results saved to {filename}")
    
    @staticmethod
    def load_results(filename='experiment_results.json'):
        import json
        
        with open(filename, 'r') as f:
            results = json.load(f)
        
        print(f"✓ Loaded {len(results)} results from {filename}")
        return results

    @staticmethod
    def plot_time_vs_size(results, operation='insert', pattern='random'):
        """
        Graph 1 & 2: Line plot of time vs size for all structures.
        """
        # Filter results
        filtered = [r for r in results 
                   if r['pattern'] == pattern 
                   and r[f'{operation}_time'] is not None]
        
        # Get unique structures
        structures = sorted(set(r['structure_name'] for r in filtered))
        
        plt.figure(figsize=(10, 6))
        
        for structure in structures:
            struct_data = [r for r in filtered if r['structure_name'] == structure]
            struct_data.sort(key=lambda x: x['size'])
            
            sizes = [r['size'] for r in struct_data]
            times = [r[f'{operation}_time'] for r in struct_data]
            
            plt.plot(sizes, times, marker='o', label=structure, linewidth=2)
        
        plt.xlabel('Dataset Size', fontsize=12)
        plt.ylabel(f'{operation.capitalize()} Time (seconds)', fontsize=12)
        plt.title(f'{operation.capitalize()} Time vs Size ({pattern} pattern)', fontsize=14)
        plt.xscale('log')
        plt.yscale('log')
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=10)
        plt.tight_layout()
        plt.savefig(f'{operation}_vs_size_{pattern}.png', dpi=300)
        plt.close()
        print(f"✓ Saved {operation}_vs_size_{pattern}.png")
    
    @staticmethod
    def plot_time_vs_pattern(results, operation='insert', size=10000):
        """
        Graph 3 & 4: Grouped bar chart of time vs pattern for all structures.
        """
        # Filter results
        filtered = [r for r in results 
                   if r['size'] == size 
                   and r[f'{operation}_time'] is not None]
        
        # Get unique patterns and structures
        patterns = sorted(set(r['pattern'] for r in filtered))
        structures = sorted(set(r['structure_name'] for r in filtered))
        
        # Prepare data
        data = {struct: [] for struct in structures}
        for pattern in patterns:
            for struct in structures:
                result = next((r for r in filtered 
                             if r['pattern'] == pattern 
                             and r['structure_name'] == struct), None)
                time_val = result[f'{operation}_time'] if result else 0
                data[struct].append(time_val)
        
        # Plotting
        fig, ax = plt.subplots(figsize=(12, 6))
        x = list(range(len(patterns)))
        width = 0.25
        
        for i, (struct, times) in enumerate(data.items()):
            offset = width * (i - 1)
            shifted_x = [xi + offset for xi in x]
            ax.bar(shifted_x, times, width, label=struct)
        
        ax.set_xlabel('Insertion Pattern', fontsize=12)
        ax.set_ylabel(f'{operation.capitalize()} Time (seconds)', fontsize=12)
        ax.set_title(f'{operation.capitalize()} Time by Pattern (Size = {size:,})', fontsize=14)
        ax.set_xticks(x)
        ax.set_xticklabels(patterns, rotation=15, ha='right')
        ax.legend(fontsize=10)
        ax.grid(True, axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{operation}_vs_pattern_size{size}.png', dpi=300)
        plt.close()
        print(f"✓ Saved {operation}_vs_pattern_size{size}.png")
    
    @staticmethod
    def generate_all_visualizations(results, fixed_pattern='random', fixed_size=10000):
        """
        Generate all 4 core graphs.
        """
        print("\n" + "="*70)
        print("GENERATING VISUALIZATIONS")
        print("="*70)
        
        # Graphs 1 & 2: Time vs Size
        ExperimentalFramework.plot_time_vs_size(results, 'insert', fixed_pattern)
        ExperimentalFramework.plot_time_vs_size(results, 'search', fixed_pattern)
        
        # Graphs 3 & 4: Time vs Pattern
        ExperimentalFramework.plot_time_vs_pattern(results, 'insert', fixed_size)
        ExperimentalFramework.plot_time_vs_pattern(results, 'search', fixed_size)
        
        print("="*70)
        print("✓ All visualizations generated!")
        print("="*70)

Use the cell below to illustrate the python code you used to **fully evaluate** your three chosen search data structures and algortihms. The code below should illustrate, for example, how you made used of the **TestDataGenerator** class to generate test data of various size and properties; how you instatiated the **ExperimentalFramework** class to  evaluate each data structure using such data, collect information about their execution time, plot results, etc. Any results you illustrate in the companion PDF report should have been generated using the code below.

In [31]:
if __name__ == "__main__":
    
    structure_classes = [ScapegoatTree, AVLTree, TwoThreeTree]
    sizes = [100, 1000]  # can add more sizes later if performance allows
    patterns = ['random', 'sorted', 'reverse_sorted', 'nearly_sorted']
    
    results = ExperimentalFramework.run_all_experiments(
        structure_classes=structure_classes,
        sizes=sizes,
        patterns=patterns,
        base_seed=84763,
        runs=5  # single run for speed
    )
    
    ExperimentalFramework.save_results(results)
    
    # load results without re-running experiments
    results = ExperimentalFramework.load_results()

    ExperimentalFramework.generate_all_visualizations(
        results,
        fixed_pattern='random',  # For graphs 1 & 2
        fixed_size=1000         # For graphs 3 & 4
    )

RUNNING EXPERIMENTS
Runs per experiment: 5

SIZE = 100

Pattern: random
------------------------------------------------------------
  Run 1/5 (seed=84763)
  Run 2/5 (seed=84764)
  Run 3/5 (seed=84765)
  Run 4/5 (seed=84766)
  Run 5/5 (seed=84767)
[1/24] ScapegoatTree ✓ Insert: 0.002291s Search: 0.000371s
[2/24] AVLTree ✓ Insert: 0.008856s Search: 0.000381s
[3/24] TwoThreeTree ✓ Insert: 0.002838s Search: 0.000511s

Pattern: sorted
------------------------------------------------------------
  Run 1/5 (seed=84763)
  Run 2/5 (seed=84764)
  Run 3/5 (seed=84765)
  Run 4/5 (seed=84766)
  Run 5/5 (seed=84767)
[4/24] ScapegoatTree ✓ Insert: 0.005828s Search: 0.002163s
[5/24] AVLTree ✓ Insert: 0.003025s Search: 0.001033s
[6/24] TwoThreeTree ✓ Insert: 0.005030s Search: 0.000943s

Pattern: reverse_sorted
------------------------------------------------------------
  Run 1/5 (seed=84763)
  Run 2/5 (seed=84764)
  Run 3/5 (seed=84765)
  Run 4/5 (seed=84766)
  Run 5/5 (seed=84767)
[7/24] ScapegoatTr